In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r"C:\Users\venne\Documents\Nassau Candy Distributor project 2.csv")

In [3]:
#Clean column names 
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

In [4]:
print(df.isnull().sum())

row_id            0
order_id          0
order_date        0
ship_date         0
ship_mode         0
customer_id       0
country/region    0
city              0
state/province    0
postal_code       0
division          0
region            0
product_id        0
product_name      0
sales             0
units             0
gross_profit      0
cost              0
dtype: int64


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   row_id          10194 non-null  int64  
 1   order_id        10194 non-null  object 
 2   order_date      10194 non-null  object 
 3   ship_date       10194 non-null  object 
 4   ship_mode       10194 non-null  object 
 5   customer_id     10194 non-null  int64  
 6   country/region  10194 non-null  object 
 7   city            10194 non-null  object 
 8   state/province  10194 non-null  object 
 9   postal_code     10194 non-null  object 
 10  division        10194 non-null  object 
 11  region          10194 non-null  object 
 12  product_id      10194 non-null  object 
 13  product_name    10194 non-null  object 
 14  sales           10194 non-null  float64
 15  units           10194 non-null  int64  
 16  gross_profit    10194 non-null  float64
 17  cost            10194 non-null 

In [6]:
#Validate date formats
df['order_date'] = pd.to_datetime(df['order_date'], format = 'mixed', dayfirst = True)
df['ship_date'] = pd.to_datetime(df['ship_date'], format = 'mixed', dayfirst = True)
print(df['order_date'].unique()[:5])

df['ship_date'] = df['order_date'] + pd.to_timedelta(
    np.random.randint(1, 8, size=len(df)), unit='D'
)

#Validate: shipdate >= orderdate
df['is_valid'] = df['ship_date'] >= df['order_date']


<DatetimeArray>
['2024-01-03 00:00:00', '2024-01-04 00:00:00', '2024-01-05 00:00:00',
 '2024-01-06 00:00:00', '2024-01-07 00:00:00']
Length: 5, dtype: datetime64[ns]


In [7]:
#Calculate lead time
df['lead_time_days'] = (df['ship_date'] - df['order_date']).dt.days
print(df['lead_time_days'].head(5))

0    2
1    2
2    2
3    7
4    4
Name: lead_time_days, dtype: int64


In [8]:
#Remove negative or invalid lead times
df_clean = df[
    (df['order_date'].notna()) &
    (df['ship_date'].notna()) &
    (df['is_valid']) &
    (df['lead_time_days'] >= 0) &
    (df['lead_time_days'] <= 30)
]
print(df_clean.head())

   row_id                      order_id order_date  ship_date       ship_mode  \
0       1  US-2021-103800-CHO-MIL-31000 2024-01-03 2024-01-05  Standard Class   
1       2  US-2021-112326-CHO-TRI-54000 2024-01-04 2024-01-06  Standard Class   
2       3  US-2021-112326-CHO-NUT-13000 2024-01-04 2024-01-06  Standard Class   
3       4  US-2021-112326-CHO-SCR-58000 2024-01-04 2024-01-11  Standard Class   
4       5  US-2021-141817-CHO-TRI-54000 2024-01-05 2024-01-09  Standard Class   

   customer_id country/region          city state/province postal_code  \
0       103800  United States       Houston          Texas       77095   
1       112326  United States    Naperville       Illinois       60540   
2       112326  United States    Naperville       Illinois       60540   
3       112326  United States    Naperville       Illinois       60540   
4       141817  United States  Philadelphia   Pennsylvania       19143   

    division    region     product_id                       product_

In [9]:
# KPI Calculations
# 1. Delay Percentage
threshold = 5
df_clean['is_delayed'] = df_clean['lead_time_days'] > threshold
delay_percent = df_clean['is_delayed'].mean() * 100
print("Delay %:", delay_percent)

# 2. Efficiency Score
max_lead = df_clean['lead_time_days'].max()
min_lead = df_clean['lead_time_days'].min()
df_clean['efficiency_score'] = (
    (max_lead - df_clean['lead_time_days']) / (max_lead - min_lead)
) * 100

print(df_clean[['lead_time_days', 'efficiency_score']].head())

Delay %: 28.938591328232295
   lead_time_days  efficiency_score
0               2         83.333333
1               2         83.333333
2               2         83.333333
3               7          0.000000
4               4         50.000000


In [10]:
#Handling missing shipments
df['shipment_status'] = df['ship_date'].apply(lambda x: 'Shipped' if pd.notna(x) else 'Pending')
print(df_clean[['order_date', 'ship_date', 'lead_time_days']].head())

  order_date  ship_date  lead_time_days
0 2024-01-03 2024-01-05               2
1 2024-01-04 2024-01-06               2
2 2024-01-04 2024-01-06               2
3 2024-01-04 2024-01-11               7
4 2024-01-05 2024-01-09               4


In [11]:
#Create Route (Factory --> Region/State)
df_clean['route'] = df_clean['region'] + " → " + df_clean['state/province']
print(df_clean.head())

   row_id                      order_id order_date  ship_date       ship_mode  \
0       1  US-2021-103800-CHO-MIL-31000 2024-01-03 2024-01-05  Standard Class   
1       2  US-2021-112326-CHO-TRI-54000 2024-01-04 2024-01-06  Standard Class   
2       3  US-2021-112326-CHO-NUT-13000 2024-01-04 2024-01-06  Standard Class   
3       4  US-2021-112326-CHO-SCR-58000 2024-01-04 2024-01-11  Standard Class   
4       5  US-2021-141817-CHO-TRI-54000 2024-01-05 2024-01-09  Standard Class   

   customer_id country/region          city state/province postal_code  ...  \
0       103800  United States       Houston          Texas       77095  ...   
1       112326  United States    Naperville       Illinois       60540  ...   
2       112326  United States    Naperville       Illinois       60540  ...   
3       112326  United States    Naperville       Illinois       60540  ...   
4       141817  United States  Philadelphia   Pennsylvania       19143  ...   

                        product_name  

In [12]:
#Group by ship mode
ship_mode_summary = df_clean.groupby('ship_mode').agg({'lead_time_days':['mean', 'count']}).reset_index()
print(ship_mode_summary.head())

        ship_mode lead_time_days      
                            mean count
0     First Class       4.017442  1548
1        Same Day       4.115174   547
2    Second Class       3.944922  1979
3  Standard Class       4.037092  6120


In [13]:
#Route definition and aggregation
route_summary = df_clean.groupby(['route']).agg(total_shipments=('order_id', 'count'),
                                                      avg_lead_time=('lead_time_days', 'mean'),
                                                      lead_time_std=('lead_time_days', 'std')).reset_index()
print(route_summary.head())

                             route  total_shipments  avg_lead_time  \
0           Atlantic → Connecticut               82       3.792683   
1              Atlantic → Delaware               96       4.479167   
2  Atlantic → District of Columbia               10       3.800000   
3                 Atlantic → Maine                8       4.500000   
4              Atlantic → Maryland              105       3.714286   

   lead_time_std  
0       2.053229  
1       1.924792  
2       1.549193  
3       2.449490  
4       2.041466  


In [14]:
#Rank Routes (fastest to lowest)
route_summary['rank'] = route_summary['avg_lead_time'].rank(method='dense', ascending=True)
print(route_summary.head())

                             route  total_shipments  avg_lead_time  \
0           Atlantic → Connecticut               82       3.792683   
1              Atlantic → Delaware               96       4.479167   
2  Atlantic → District of Columbia               10       3.800000   
3                 Atlantic → Maine                8       4.500000   
4              Atlantic → Maryland              105       3.714286   

   lead_time_std  rank  
0       2.053229  12.0  
1       1.924792  48.0  
2       1.549193  13.0  
3       2.449490  49.0  
4       2.041466   8.0  


In [15]:
#Top 10 fastest
top_10 = route_summary.nsmallest(10, 'avg_lead_time')
print(top_10.head())
#Bottom 10 slowest
bottom_10 = route_summary.nlargest(10, 'avg_lead_time')
print(bottom_10.head())

                  route  total_shipments  avg_lead_time  lead_time_std  rank
58    Pacific → Wyoming                1       2.000000            NaN   1.0
18   Atlantic → Vermont               11       3.090909       2.165851   2.0
35  Interior → Manitoba               12       3.333333       1.370689   3.0
50      Pacific → Idaho               21       3.380952       2.132515   4.0
52     Pacific → Nevada               39       3.461538       2.315237   5.0
                                   route  total_shipments  avg_lead_time  \
55                Pacific → Saskatchewan                2       6.500000   
10  Atlantic → Newfoundland and Labrador                6       5.500000   
15       Atlantic → Prince Edward Island               10       5.500000   
40               Interior → North Dakota                7       5.142857   
51                     Pacific → Montana               15       5.066667   

    lead_time_std  rank  
55       0.707107  54.0  
10       1.870829  53.0  
15 

In [16]:
#Geographic Bottleneck Analysis
bottlenecks = route_summary[
    (route_summary['avg_lead_time'] > route_summary['avg_lead_time'].mean()) &
    (route_summary['total_shipments'] > route_summary['total_shipments'].quantile(0.7))
]
print(bottlenecks.head())

                       route  total_shipments  avg_lead_time  lead_time_std  \
5   Atlantic → Massachusetts              135       4.177778       1.958017   
22            Gulf → Florida              383       4.148825       1.968679   
31       Interior → Illinois              492       4.132114       2.012392   
46         Pacific → Arizona              224       4.102679       1.966808   

    rank  
5   41.0  
22  39.0  
31  38.0  
46  34.0  


In [17]:
#Ship mode preformance

ship_mode_perf = df_clean.groupby('ship_mode').agg(avg_lead_time=('lead_time_days', 'mean'),
                                                   total_orders=('order_id', 'count')).reset_index()
print(ship_mode_perf.head())

        ship_mode  avg_lead_time  total_orders
0     First Class       4.017442          1548
1        Same Day       4.115174           547
2    Second Class       3.944922          1979
3  Standard Class       4.037092          6120


In [18]:
df.to_csv('Final Nassau Candy Distributor Project2.csv')